# VIST-NRF — Main Execution Notebook

End-to-end pipeline: mount Drive, clone/pull the repo, install the package, download PhysioNet data via KaggleHub, build targets/features/splits, train a model, evaluate, and save results to Drive.

Prerequisite: add your GitHub PAT to Colab Secrets (left sidebar → key icon) as `GITHUB_TOKEN`.

In [ ]:
import os, stat, subprocess, shutil, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['VIST_NRF_DRIVE'] = '/content/drive/MyDrive/VIST-NRF'
os.environ['VIST_NRF_DATA'] = str(Path(os.environ['VIST_NRF_DRIVE']) / 'data')
os.environ['VIST_NRF_RESULTS'] = str(Path(os.environ['VIST_NRF_DRIVE']) / 'results')
for key in ('VIST_NRF_DATA', 'VIST_NRF_RESULTS'):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

repo_url = "https://github.com/avan-codes/VIST-NRF.git"
repo_dir = "/content/VIST-NRF"
token = userdata.get('GITHUB_TOKEN')

if not token:
    raise RuntimeError('Colab Secrets mein GITHUB_TOKEN add karein (left sidebar → key icon).')

# Ensure that if the directory exists but is not a valid git repository, we clean it up first
if Path(repo_dir).exists() and not (Path(repo_dir) / '.git').exists():
    print(f"Removing non-git or incomplete directory: {repo_dir}")
    shutil.rmtree(repo_dir)

askpass = Path('/tmp/vist_nrf_askpass.sh')
askpass.write_text('#!/bin/sh\\ncase "$1" in\\n*Username*) echo x-access-token ;;&\\n*Password*) printf "%s" "$GITHUB_TOKEN" ;;&\\nesac\\n')
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
env = os.environ.copy()
env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', GITHUB_TOKEN=token)
try:
    if not Path(repo_dir).exists():
        print("Cloning repository...")
        result = subprocess.run(['git', 'clone', repo_url, repo_dir], env=env, capture_output=True, text=True)
        if result.returncode != 0:
            if "403" in result.stderr or "Write access" in result.stderr:
                raise PermissionError("GitHub Access Denied (403): Please check your GITHUB_TOKEN permissions, validity, or repository scopes in GitHub Settings.")
            print("Git Clone Standard Error:", result.stderr)
            result.check_returncode()
    else:
        print("Pulling updates...")
        result = subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], env=env, capture_output=True, text=True)
        if result.returncode != 0:
            print("Git Pull Standard Error:", result.stderr)
            result.check_returncode()
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN', None)
    if 'token' in locals():
        del token

In [ ]:
# Install using the active environment's interpreter explicitly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo_dir], check=True)

# Register both root and src subdirectory to sys.path to guarantee discovery
for p in (repo_dir, str(Path(repo_dir) / 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

In [ ]:
import torch, vist_nrf
commit = subprocess.check_output(['git', '-C', repo_dir, 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None', '| commit:', commit)

In [ ]:
import kagglehub, os
path = kagglehub.dataset_download("farjanayesmin/the-physionet-challenge-2019-dataset")
print('Dataset path:', path)
os.environ['VIST_NRF_DATA_RAW'] = path

In [ ]:
# Suppress pandas PerformanceWarning from feature history fragmentation
import warnings
import pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import sys, os, json
from pathlib import Path

# Deep clean sys.path to ensure no stray /content/VIST-NRF is shadowing /content/VIST-NRF/src
repo_root_path = '/content/VIST-NRF'
src_path = '/content/VIST-NRF/src'

# Clean all occurrences of both paths
sys.path = [p for p in sys.path if p != repo_root_path and p != src_path]

# Insert the source path at the absolute front
sys.path.insert(0, src_path)

# Forcibly purge all cached 'vist_nrf' entries in sys.modules
for key in list(sys.modules.keys()):
    if key == 'vist_nrf' or key.startswith('vist_nrf.'):
        sys.modules.pop(key, None)

# Verify where Python is importing vist_nrf from
import vist_nrf
print("vist_nrf imported from:", getattr(vist_nrf, '__file__', 'unknown'))

import numpy as np
import yaml

# Perform imports now that the namespace is correctly pointed and fully cleared
from vist_nrf.utils.seed import set_seed
from vist_nrf.utils.io import environment_metadata, git_commit, save_jsonl, utc_now
from vist_nrf.utils.prereg import enforce_prereg, prereg_hash, PreregistrationError
try:
    from vist_nrf.data.physionet2019 import load_environment
    from vist_nrf.data.splits import split_by_environment, assert_splits_valid
    from vist_nrf.data.targets import build_targets
except ModuleNotFoundError as e:
    print(f"\n[ERROR] Missing Module: {e}")
    print("The subpackage 'vist_nrf.data' is missing from the repository clone.")
    print("Please verify if the 'data' module has been pushed to your GitHub repository.")
    raise e

from vist_nrf.features.history import FeatureConfig, build_feature_matrix, assert_causal
from vist_nrf.eval.censoring import censoring_report, assert_no_early_event_prediction, assert_censoring_not_negative
from vist_nrf.eval.metrics import evaluate
from vist_nrf.models import build_model

set_seed(0)
repo_root = Path('/content/VIST-NRF')
commit = git_commit(repo_root)
meta = environment_metadata()
meta.update({'commit': commit, 'utc': utc_now()})

# Config load
cfg_path = '/content/VIST-NRF/configs/baselines/e1_a_to_a.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Preregistration (exploratory mode since not frozen)
enforce_prereg(repo_root, allow_exploratory=True)

raw_root = os.environ.get('VIST_NRF_DATA_RAW', cfg['dataset']['raw_root'])
train_env = cfg.get('train_env', 'A')
test_envs = cfg.get('test_envs', ['A'])
horizon_hours = int(cfg.get('horizon_hours', 12))
min_history = int(cfg.get('min_history_hours', 6))

print(f"Loading real PhysioNet data from {raw_root}")
patient_frames = {}
for env in [train_env] + test_envs:
    env_frames = load_environment(raw_root, env)
    patient_frames.update(env_frames)
    print(f"  env {env}: {len(env_frames)} patients")

# Targets
targets_long, patient_summary = build_targets(
    patient_frames, label_col="SepsisLabel",
    horizon_hours=horizon_hours, min_history_hours=min_history,
)
assert_no_early_event_prediction(targets_long)
assert_censoring_not_negative(targets_long)
cens_report = censoring_report(targets_long, patient_summary)
print(f"Censoring: {cens_report['n_censored_rows']} rows censored, {cens_report['n_eligible_rows']} eligible")

# Features
clinical_cols = [c for c in patient_frames[next(iter(patient_frames))].columns
                 if c not in ("SepsisLabel", "hour", "patient_id", "Age", "Gender",
                              "HospAdmTime", "ICULOS", "ICU_unit", "_source_file")]
static_cols = [c for c in ("Age", "Gender", "HospAdmTime", "ICULOS", "ICU_unit")
               if c in patient_frames[next(iter(patient_frames))].columns]
cfg_feat = FeatureConfig(clinical_cols=clinical_cols, static_cols=static_cols,
                         min_history_hours=min_history,
                         patient_id_col="patient_id", hour_col="hour")

feature_matrix = build_feature_matrix(patient_frames, cfg_feat)
assert_causal(feature_matrix, patient_frames, cfg_feat)
print("Causality / leakage test: PASSED")

# Splits (patient-level, 70/15/15)
splits = split_by_environment(
    {e: [p for p in patient_frames if p.startswith(e)] for e in [train_env] + test_envs},
    {e: {p: 1 for p in patient_frames if p.startswith(e)} for e in [train_env] + test_envs},
    seed=0,
)
assert_splits_valid(splits)
print(f"Splits: {{e: (len(s.train), len(s.val), len(s.test)) for e, s in splits.items()}}")

# Model
model_cfg = cfg.get("model", {})
model = build_model(model_cfg.get("name", "logistic_regression"), seed=0,
                    **dict(model_cfg.get("params", {})))

# Train on source env train split
train_pids = splits[train_env].train
tr = targets_long.merge(
    feature_matrix[["patient_id", "hour"] + [c for c in feature_matrix.columns
                                               if c not in ("patient_id", "hour")]],
    on=["patient_id", "hour"], how="inner",
)
tr = tr[tr["eligible_for_scoring"]]
feature_cols = [c for c in tr.columns if c not in ("patient_id", "hour", "y", "censored", "eligible_for_scoring")]
Xtr = tr[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
ytr = tr["y"].to_numpy(dtype=int)
env_tr = np.zeros(len(ytr), dtype=int)

print(f"Training {model_cfg.get('name')} on {len(Xtr)} rows from {len(train_pids)} patients (env {train_env})")
model.fit(Xtr, ytr, envs=env_tr)

# Evaluate on test env(s)
results = []
for test_env in test_envs:
    test_pids = splits[test_env].test
    te_mask = feature_matrix["patient_id"].isin(test_pids)
    te_fm = feature_matrix.loc[te_mask]
    te = targets_long.merge(
        te_fm[["patient_id", "hour"] + [c for c in te_fm.columns
                                         if c not in ("patient_id", "hour")]],
        on=["patient_id", "hour"], how="inner",
    )
    te = te[te["eligible_for_scoring"]]
    Xte = te[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    yte = te["y"].to_numpy(dtype=int)
    p = model.predict_proba(Xte)[:, 1]
    m = evaluate(yte, p)
    print(f"  {test_env}: AUROC={m.auroc:.3f} AUPRC={m.auprc:.3f} Brier={m.brier:.3f} ECE={m.ece:.3f} prev={m.prevalence:.3f} n={m.n}")

    rec = {"train_env": train_env, "test_env": test_env, "model": model_cfg.get("name"),
           "horizon_hours": horizon_hours, "seed": 0, "commit": commit,
           "n_test_patients": len(test_pids), "n_test_rows": len(Xte), **m.__dict__,
           "censored_rows": cens_report["n_censored_rows"]}
    rec.update(meta)
    results.append(rec)

# Save to Drive
results_dir = Path(os.environ['VIST_NRF_RESULTS']) / "runs"
results_dir.mkdir(parents=True, exist_ok=True)
for rec in results:
    with open(results_dir / f"e1_lr_a_to_a_seed0.jsonl", "a") as f:
        f.write(json.dumps(rec, sort_keys=True) + "\n")
print("Results saved to Drive:", results_dir)